# Performance, Complexity and Explanation Stability in P2P Credit-Risk Models

**What this notebook does.** It is a thin driver: all logic lives in `src/`, and each section below
calls one module and shows its output. In order it (1) sets up the configuration and a chronological
train / test split, (2) audits the data cleaning, (3) fits every model for every seed, (4) tests H1
(are models performance-equivalent?), (5) tests H2 and H3 (do explanations differ beyond seed noise?),
(6) checks calibration, (7) runs drift, stress and backtest checks, (8) compares runs with and without
the lender's own `int_rate` / `sub_grade`, and (9) builds the scorecard and trade-off plots.

Set `USE_SYNTHETIC = False` and place the LendingClub CSV in `data/raw/` (see `data/README.md`) for the
real analysis. With synthetic data the numbers only prove the pipeline runs.

**Hypotheses.** H1: model pairs are performance-equivalent (PR-AUC, TOST) or not. H2: explanations
shift between models by more than seed-to-seed variation. H3: some pair is equivalent in performance
but explained differently.

### 0. Setup
Imports the package, silences warnings and chooses synthetic or real data.

In [ ]:
import sys, warnings
from dataclasses import replace
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd

from src.config import ExperimentConfig, USECOLS
from src.experiments import (load_frame, load_splits, run_seed_grid, run_hypothesis_tests, summarise,
                             save_tables, run_feature_set_comparison)
from src.models import available_models

warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)

USE_SYNTHETIC = True        # True = generated data (pipeline check only); False = real LendingClub CSV
COMPARE_FEATURE_SETS = True # section 7 re-runs everything twice; set False to skip on large real runs
print("Models available in this environment:", available_models())

### 1. Configuration and chronological split
Builds the settings object, then cleans the data and splits it by origination date (train = earlier months, test = latest months).

In [ ]:
cfg = ExperimentConfig()
if USE_SYNTHETIC:
    cfg = replace(cfg, seeds=tuple(range(4)), seed_train_size=3_000, explain_size=400,
                  n_boot=100, n_boot_explain=300, top_k=5,
                  output_dir=ROOT / "outputs" / "tables" / "synthetic_demo")
else:
    cfg = replace(cfg, models=("Logistic Regression", "Random Forest", "XGBoost"))

splits = load_splits(cfg, synthetic=USE_SYNTHETIC)
X_train, y_train, X_test, y_test = splits
print(f"lender score dropped: {cfg.drop_lender_assigned}")
print(f"train {len(X_train):,} rows, default rate {y_train.mean():.2%}")
print(f"test  {len(X_test):,} rows, default rate {y_test.mean():.2%}  (later origination months)")

### 2. Data-quality audit
Shows how many loans survive each cleaning step, which loan statuses were dropped, the default rate by issue year (a sharp fall in the latest years means right-censoring) and the most-missing features. See `docs/data_cleaning.md` for the reasons.

In [ ]:
from src.data_quality import data_quality_report
from src.synthetic import make_synthetic_lendingclub

raw = (make_synthetic_lendingclub(8_000, seed=0) if USE_SYNTHETIC
       else pd.read_csv(cfg.csv_path, usecols=USECOLS, low_memory=False))
report = data_quality_report(raw, cfg, verbose=False)
display(report["attrition"])
display(report["status_breakdown"])
display(report["default_rate_by_year"])
display(report["missingness"].head(10))

### 3. Seed grid
Fits every model for every seed on a different training subsample, scores the same test rows and computes one global importance vector per fit. Then summarises mean performance, calibration, generalization gap and explanation stability per model, and the structural size of each model.

In [ ]:
grid = run_seed_grid(splits, cfg, verbose=False)
summary = summarise(grid)
display(summary.round(4))
display(grid["complexity"])

### 4. H1 - performance equivalence
For every model pair, bootstraps the PR-AUC difference (resampling seeds and test rows) and compares its 90% interval with the equivalence margin. Verdicts: *equivalent*, *different* or *inconclusive* (data cannot tell).

In [ ]:
tests = run_hypothesis_tests(grid, cfg, verbose=False)
display(tests["h1"].round(4))

from src.visualization import plot_equivalence_forest
plot_equivalence_forest(tests["h1"]); plt.show()

### 5. H2 and H3 - explanations
H2 asks whether two models' explanations (top-k membership and ranking) differ more than the same model's explanations differ across seeds. H3 flags pairs that are performance-equivalent but explained differently.

In [ ]:
display(tests["h2"].round(4))
display(tests["h3"])

from src.visualization import plot_explanation_shift
plot_explanation_shift(tests["h2"]); plt.show()

### 6. Calibration
Compares Brier score and ECE before and after isotonic recalibration, and draws the reliability curves (closer to the diagonal is better).

In [ ]:
from src.calibration import compare_calibration
from src.visualization import plot_calibration

sub = X_train.sample(min(len(X_train), 20_000), random_state=0)
calib = compare_calibration(grid["fitted"], sub, y_train.loc[sub.index], X_test, y_test, methods=("isotonic",))
display(calib.round(4))
plot_calibration(grid["probs"], grid["y_test"]); plt.show()

### 7. Robustness
PSI measures how far each feature drifted between train and test; the stress table shows mean predicted PD under worse income / DTI / utilisation (it should rise); the backtest trains on earlier years and tests on each later year.

In [ ]:
from src.robustness import feature_psi, stress_test

display(feature_psi(X_train, X_test).head(10).round(4))

stress = {}
for name, pipe in grid["fitted"].items():
    stress[name] = stress_test(lambda d, p=pipe: p.predict_proba(d)[:, 1], X_test.head(2_000)) \
        .set_index("Scenario")["Average_PD"]
display(pd.DataFrame(stress).round(4))

In [ ]:
from src.robustness import chronological_backtest
from src.models import make_pipeline
from src.preprocessing import prepare_frame

# The backtest needs the dated frame (the split above removed the dates), so rebuild it.
frame = prepare_frame(raw, sample_size=None if USE_SYNTHETIC else cfg.sample_size,
                      max_issue_year=cfg.max_issue_year, drop_lender_assigned=cfg.drop_lender_assigned)
bt = chronological_backtest(lambda X, y: make_pipeline("Logistic Regression", X), frame,
                            first_test_year=2014, min_train=500 if USE_SYNTHETIC else 3_000,
                            min_test=200 if USE_SYNTHETIC else 500)
display(bt.round(4))

### 8. With vs without the lender's own score
Runs the whole experiment twice - without and with `int_rate` / `sub_grade` - to show how much of each model's performance is just LendingClub's own underwriting score.

In [ ]:
if COMPARE_FEATURE_SETS:
    both = run_feature_set_comparison(cfg, synthetic=USE_SYNTHETIC, verbose=False)
    display(both["comparison"][["roc_auc", "pr_auc", "brier", "ece", "explanation_stability"]].round(4))
else:
    print("Skipped (COMPARE_FEATURE_SETS = False).")

### 9. Scorecard and trade-offs
Turns the measured results into relative 0-100 scores per model (radar chart), plots performance against complexity, and lists Pareto-dominated models. Keep the raw tables above next to these scores: they only rank the compared models against each other.

In [ ]:
from src.scoring import build_scorecard, complexity_score, find_dominated_models
from src.visualization import plot_radar, plot_tradeoff

scorecard = build_scorecard(summary, grid["complexity"])
display(scorecard.round(1))
plot_radar(scorecard); plt.show()

table = summary.join(scorecard)
table["Complexity_Score"] = complexity_score(scorecard)
plot_tradeoff(table, "Complexity_Score", "pr_auc", "Complexity score (100 - Efficiency)",
              "Mean test PR-AUC", "Performance vs complexity", min_y_span=0.05); plt.show()

print(find_dominated_models(table, maximize=["pr_auc", "explanation_stability"], minimize=["Complexity_Score"]))
save_tables(cfg.output_dir, grid, tests, summary, scorecard)

### 10. Robustness and the full figure set
Runs drift (PSI), sensitivity, stress and the chronological backtest, then draws every remaining figure (ROC, confusion matrices, threshold trade-off, importance bars, seed boxplots, rank agreement, robustness charts, trade-off dashboard) and saves them to `outputs/figures/`. Sensitivity and stress shocks are illustrative, not macro-calibrated.

In [ ]:
from src.experiments import run_robustness
from src.visualization import save_all

frame = load_frame(cfg, USE_SYNTHETIC)
cfg_fig = replace(cfg, backtest_first_year=2013, backtest_min_train=500, backtest_min_test=100) if USE_SYNTHETIC else cfg
robustness = run_robustness(grid, splits, frame, cfg_fig, verbose=False)
result = {"grid": grid, "tests": tests, "summary": summary, "scorecard": scorecard, "robustness": robustness}
paths = save_all(result, ROOT / "outputs" / "figures")
print(f"Wrote {len(paths)} figures to outputs/figures/")
from IPython.display import Image, display as _d
for p in paths:
    _d(Image(filename=str(p), width=800))

### Reading the results

- An *inconclusive* H1 verdict means the data cannot show equivalence **or** a difference; it is not evidence of "no difference".
- H3 is only meaningful when H1 says *equivalent* and H2 survives Holm correction.
- SHAP explains model behaviour, not causes. The scorecard is relative: keep the raw tables next to it.